# 資料前處理程式練習 (12 題)
使用隨附的 customers.csv，共 240 筆教學合成資料。請將 CSV 和 Notebook 放在同一個資料夾，依題號執行；第 3 題起會沿用前題變數。教師解答可以重新啟動 Kernel 後由上到下執行。

環境：Python、pandas、NumPy、scikit-learn (>=1.2)、Jupyter、joblib。安裝指令：`uv add pandas numpy scikit-learn jupyter joblib`。

| 欄位 | 意義 |
|---|---|
| customer_id | 客戶 ID，不納入模型特徵 |
| age | 年齡 (歲) |
| monthly_income | 月收入 (元) |
| website_visits | 最近一個月網站造訪次數 |
| city | 城市 (無順序類別) |
| channel | 接觸管道 (無順序類別) |
| membership | 會員等級，Basic < Silver < Gold |
| purchased | 購買結果，No / Yes (Target) |

共通規則：先切分資料，才學習補值、編碼與縮放規則。測試集與新客戶只使用 transform / predict。每題除了程式，也請確認指定的檢查結果。


## 01｜讀取與資料品質報告

讀取 customers.csv 成為 df。輸出資料形狀、前 5 筆、每欄型態；建立包含「缺失筆數」與「缺失比例」的表格 (比例取小數第 3 位)。輸出完全重複列數、重複 ID 數，以及 purchased 各類別筆數。

檢查：形狀應為 (240, 8)；Yes 124 筆、No 116 筆。age 缺失 21 筆、monthly_income 24 筆。

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

### 讀取資料與檢查基本資訊
df = pd.read_csv(Path("../data/customers.csv")) # CSV 與 Notebook 放在同一個資料夾
print("資料形狀：", df.shape)
display(df.head(5)) # 顯示前 5 筆客戶資料
display(df.dtypes) # 查看各欄位的資料型態

### 建立缺失值報告
report = pd.DataFrame({"缺失筆數": df.isna().sum(), "缺失比例": df.isna().mean()})
display(report.round(3)) # 比例使用 0～1 表示，顯示到小數第 3 位
print("完全重複列：", df.duplicated().sum()) # 不計第一次出現的資料
print("重複 ID：", df["customer_id"].duplicated().sum())
display(df["purchased"].value_counts(dropna=False)) # 包含缺失答案的統計

資料形狀： (240, 8)


,customer_id,age,monthly_income,website_visits,city,channel,membership,purchased
0,C0001,60.0,38000.0,0.0,Tainan,NaN,Basic,No
1,C0002,52.0,101000.0,0.0,Tainan,Ads,NaN,Yes
2,C0003,30.0,NaN,5.0,Taichung,Ads,Basic,Yes
3,C0004,54.0,39000.0,6.0,Taipei,Referral,Silver,No
4,C0005,34.0,61000.0,1.0,Taipei,Ads,Silver,No


customer_id        object
age               float64
monthly_income    float64
website_visits    float64
city               object
channel            object
membership         object
purchased          object
dtype: object

,缺失筆數,缺失比例
customer_id,0,0.000
age,21,0.088
monthly_income,24,0.100
website_visits,6,0.025
city,7,0.029
channel,18,0.075
membership,10,0.042
purchased,0,0.000


完全重複列： 0
重複 ID： 0


purchased
Yes    124
No     116
Name: count, dtype: int64

## 02｜清理文字數字與不合理資料

使用題目提供的小表，不修改 df。移除 city 前後空白，將空字串和 ? 改為 NaN；將 age 轉為數值，無法轉換或小於 0 的年齡改為 NaN。輸出清理後表格及各欄缺失筆數。

資料：age = [" 25 ", "?", "-3", "40", "abc"]；city = [" Taipei ", "", "Taichung", " Tainan ", "?"]。
檢查：age 缺失 3 筆、city 缺失 2 筆；有效年齡為 25、40。

In [4]:
### 建立題目資料
dirty = pd.DataFrame({"age": [" 25 ", "?", "-3", "40", "abc"], "city": [" Taipei ", "", "Taichung", " Tainan ", "?"]})
clean = dirty.copy() # 複製資料，保留原始小表
clean["city"] = clean["city"].str.strip().replace({"": np.nan, "?": np.nan}) # 統一文字缺失標記
clean["age"] = pd.to_numeric(clean["age"], errors="coerce") # 無法轉換的文字改為 NaN
clean.loc[clean["age"] < 0, "age"] = np.nan # 負年齡不合理，留待補值
display(clean)
display(clean.isna().sum())

,age,city
0,25.0,Taipei
1,NaN,NaN
2,NaN,Taichung
3,40.0,Tainan
4,NaN,NaN


age     3
city    2
dtype: int64

## 03｜分出 Features 與 Target

從 df 移除 purchased 缺失的列，存成 labeled_df。數值特徵指定 age、monthly_income、website_visits；類別特徵指定 city、channel、membership。建立 X (只含這 6 欄) 與 y (purchased)。輸出 X、y 的形狀及前 3 筆。

檢查：X 為 (240, 6)，y 為 (240,)；X 不含 customer_id 或 purchased。不要補值 Target。

In [5]:
### 明確指定特徵欄位
numeric_features = ["age", "monthly_income", "website_visits"]
categorical_features = ["city", "channel", "membership"]
feature_columns = numeric_features + categorical_features
labeled_df = df.dropna(subset=["purchased"]).copy() # 缺少答案的列無法用來監督訓練
X = labeled_df[feature_columns].copy() # 排除 ID 與答案，避免錯用欄位
y = labeled_df["purchased"].copy() # 單一欄位保留為一維 Series
print("X：", X.shape, "y：", y.shape)
display(X.head(3))
display(y.head(3))

X： (240, 6) y： (240,)


,age,monthly_income,website_visits,city,channel,membership
0,60.0,38000.0,0.0,Tainan,NaN,Basic
1,52.0,101000.0,0.0,Tainan,Ads,NaN
2,30.0,NaN,5.0,Taichung,Ads,Basic


0     No
1    Yes
2    Yes
Name: purchased, dtype: object

## 04｜切分訓練集與測試集

使用 train_test_split 同時切分 X、y；設定 test_size=0.25、random_state=42、stratify=y。輸出兩組 X、y 的形狀；建立全部、訓練、測試的答案比例比較表，取小數第 3 位。

檢查：訓練 180 筆、測試 60 筆，兩者各有 6 個特徵；X_train 和 y_train 的索引必須一致。後續題目固定使用這次切分。

In [6]:
from sklearn.model_selection import train_test_split

### 同時切分輸入與答案
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y) # 維持答案類別比例
print("訓練：", X_train.shape, y_train.shape)
print("測試：", X_test.shape, y_test.shape)
ratios = pd.DataFrame({"全部": y.value_counts(normalize=True), "訓練": y_train.value_counts(normalize=True), "測試": y_test.value_counts(normalize=True)})
display(ratios.round(3))
print("訓練索引一致：", X_train.index.equals(y_train.index)) # 確認每筆資料與答案仍對應

訓練： (180, 6) (180,)
測試： (60, 6) (60,)


,全部,訓練,測試
purchased,,,
Yes,0.517,0.517,0.517
No,0.483,0.483,0.483


訓練索引一致： True


## 05｜學習補值規則

數值欄位用 SimpleImputer(strategy="median")，類別欄位用 strategy="most_frequent"。只從訓練集學習，測試集沿用規則。結果存為 train_num、test_num、train_cat、test_cat 四個 DataFrame，保留欄名與原始索引。輸出兩種補值規則及四個表的缺失總數。

檢查：四個表的缺失總數均為 0；train_num 形狀 (180, 3)，test_cat 形狀 (60, 3)。

In [7]:
from sklearn.impute import SimpleImputer

### 補數值缺失值
num_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
train_num = pd.DataFrame(num_imputer.fit_transform(X_train[numeric_features]), columns=numeric_features, index=X_train.index) # 從訓練集學中位數

test_num = pd.DataFrame(num_imputer.transform(X_test[numeric_features]), columns=numeric_features, index=X_test.index) # 沿用訓練中位數

### 補類別缺失值
cat_imputer = SimpleImputer(strategy="most_frequent", keep_empty_features=True)
train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[categorical_features]), columns=categorical_features, index=X_train.index) # 學各欄眾數
test_cat = pd.DataFrame(cat_imputer.transform(X_test[categorical_features]), columns=categorical_features, index=X_test.index)
display(pd.Series(num_imputer.statistics_, index=numeric_features, name="中位數"))
display(pd.Series(cat_imputer.statistics_, index=categorical_features, name="眾數"))
for name, table in [("train_num", train_num), ("test_num", test_num), ("train_cat", train_cat), ("test_cat", test_cat)]:
    print(name, "缺失總數：", table.isna().sum().sum()) # 加總每欄缺失筆數

age                  42.5
monthly_income    71000.0
website_visits        6.0
Name: 中位數, dtype: float64

city          Tainan
channel          Ads
membership      Gold
Name: 眾數, dtype: object

train_num 缺失總數： 0
test_num 缺失總數： 0
train_cat 缺失總數： 0
test_cat 缺失總數： 0


## 06｜編碼並還原 Target

使用 LabelEncoder 從 y_train 建立規則，轉換 y_train、y_test，存為 y_train_encoded、y_test_encoded。輸出標籤對照表；將訓練答案前 5 筆編碼再還原，比較還原結果與原始答案。

檢查：No 對應 0、Yes 對應 1；還原前 5 筆應與 y_train.iloc[:5] 相同。

In [8]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder() # 將答案名稱轉成整數代碼
y_train_encoded = label_encoder.fit_transform(y_train) # 只從訓練答案建立規則
y_test_encoded = label_encoder.transform(y_test) # 測試答案沿用相同對照
display(pd.DataFrame({"標籤": label_encoder.classes_, "代碼": range(len(label_encoder.classes_))}))
restored = label_encoder.inverse_transform(y_train_encoded[:5]) # 整數轉回原始標籤
print("原始：", y_train.iloc[:5].tolist())
print("還原：", restored.tolist())
print("還原一致：", np.array_equal(restored, y_train.iloc[:5].to_numpy()))

,標籤,代碼
0,No,0
1,Yes,1


原始： ['Yes', 'No', 'No', 'Yes', 'Yes']
還原： ['Yes', 'No', 'No', 'Yes', 'Yes']
還原一致： True


## 07｜One-Hot 編碼與未知城市

將 train_cat、test_cat 的 3 個欄位全部 One-Hot 編碼，設定 handle_unknown="ignore"、sparse_output=False；保留索引與編碼後欄名。輸出前 3 筆與形狀。建立一筆 city="Kaohsiung"、channel="Ads"、membership="Gold" 的資料，用相同編碼器轉換。

檢查：訓練、測試欄名與順序一致，各 9 欄；未知 Kaohsiung 的所有 city_ 欄位均為 0。

In [9]:
from sklearn.preprocessing import OneHotEncoder

onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False) # 未知類別不報錯，其所屬欄位的旗標皆為 0
train_encoded_array = onehot.fit_transform(train_cat) # 學習訓練類別並轉換
encoded_columns = onehot.get_feature_names_out(categorical_features)
train_cat_encoded = pd.DataFrame(train_encoded_array, columns=encoded_columns, index=X_train.index)
test_cat_encoded = pd.DataFrame(onehot.transform(test_cat), columns=encoded_columns, index=X_test.index)
display(train_cat_encoded.head(3))
print("形狀：", train_cat_encoded.shape, test_cat_encoded.shape)
unknown = pd.DataFrame({"city": ["Kaohsiung"], "channel": ["Ads"], "membership": ["Gold"]})
unknown_encoded = pd.DataFrame(onehot.transform(unknown), columns=encoded_columns)
display(unknown_encoded)
print("城市旗標總和：", unknown_encoded.filter(like="city_").sum(axis=1).iloc[0])

,city_Taichung,city_Tainan,city_Taipei,channel_Ads,channel_Organic,channel_Referral,membership_Basic,membership_Gold,membership_Silver
91,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
47,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
211,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0


形狀： (180, 9) (60, 9)


,city_Taichung,city_Tainan,city_Taipei,channel_Ads,channel_Organic,channel_Referral,membership_Basic,membership_Gold,membership_Silver
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


城市旗標總和： 0.0


## 08｜會員等級的順序編碼

獨立建立 membership=["Basic", "Gold", "Silver", "Platinum"] 的小表。使用 OrdinalEncoder 指定 Basic、Silver、Gold 的順序，未知等級編為 -1。先用前三筆 fit_transform，再用第四筆 transform；輸出原始等級與代碼。

檢查：依資料順序得到 [0, 2, 1, -1]。本題只示範順序編碼，不替換第 7 題的 One-Hot 結果。

In [10]:
from sklearn.preprocessing import OrdinalEncoder

levels = pd.DataFrame({"membership": ["Basic", "Gold", "Silver", "Platinum"]})
ordinal = OrdinalEncoder(categories=[["Basic", "Silver", "Gold"]], handle_unknown="use_encoded_value", unknown_value=-1) # 手動指定順序
known_codes = ordinal.fit_transform(levels.iloc[:3]).ravel() # 將二維單欄結果攤平成一維
unknown_codes = ordinal.transform(levels.iloc[3:]).ravel() # Platinum 使用 -1
result = levels.assign(encoded=np.concatenate([known_codes, unknown_codes])) # 合併兩組編碼，保持原始順序
display(result)

,membership,encoded
0,Basic,0.0
1,Gold,2.0
2,Silver,1.0
3,Platinum,-1.0


## 09｜比較標準化與 Min-Max 縮放

使用第 5 題的 train_num、test_num，分別建立 StandardScaler 與 MinMaxScaler；兩者都只從 train_num 學習。標準化結果存為 train_num_scaled、test_num_scaled。輸出標準化訓練集每欄平均與標準差 (ddof=0)，以及 Min-Max 訓練集每欄最小值與最大值，取小數第 3 位。

檢查：標準化訓練平均約 0、標準差約 1；Min-Max 訓練最小 0、最大 1。測試集不用符合這些數值，不要再次 fit。

In [11]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

### 標準化
standard_scaler = StandardScaler()
train_num_scaled = pd.DataFrame(standard_scaler.fit_transform(train_num), columns=numeric_features, index=X_train.index)
test_num_scaled = pd.DataFrame(standard_scaler.transform(test_num), columns=numeric_features, index=X_test.index)
display(pd.DataFrame({"平均": train_num_scaled.mean(), "標準差": train_num_scaled.std(ddof=0)}).round(3)) # 與 StandardScaler 使用相同的 ddof

### Min-Max 縮放
minmax_scaler = MinMaxScaler()
train_minmax = pd.DataFrame(minmax_scaler.fit_transform(train_num), columns=numeric_features, index=X_train.index)
test_minmax = pd.DataFrame(minmax_scaler.transform(test_num), columns=numeric_features, index=X_test.index)
display(pd.DataFrame({"最小值": train_minmax.min(), "最大值": train_minmax.max()}).round(3))

,平均,標準差
age,-0.0,1.0
monthly_income,-0.0,1.0
website_visits,0.0,1.0


,最小值,最大值
age,0.0,1.0
monthly_income,0.0,1.0
website_visits,0.0,1.0


## 10｜手動合併特徵並訓練模型

依欄方向合併 train_num_scaled 與 train_cat_encoded，測試資料也同樣合併，存為 X_train_ready、X_test_ready。用 LogisticRegression(max_iter=1000) 與文字答案 y_train 訓練，預測測試集。輸出合併形狀、是否含缺失值、Accuracy 與 classification_report。

檢查：訓練 (180, 12)、測試 (60, 12)，無缺失值；預測 60 筆 No / Yes。不要把 y 拼入特徵，不要求分數達到特定門檻。

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

### 依索引對齊後合併兩種特徵
X_train_ready = pd.concat([train_num_scaled, train_cat_encoded], axis=1) # axis=1 表示增加欄位
X_test_ready = pd.concat([test_num_scaled, test_cat_encoded], axis=1)
print("形狀：", X_train_ready.shape, X_test_ready.shape)
print("含缺失值：", X_train_ready.isna().any().any(), X_test_ready.isna().any().any())
manual_model = LogisticRegression(max_iter=1000) # 讓模型有足夠的迭代次數
manual_model.fit(X_train_ready, y_train) # sklearn 此模型可以直接使用文字答案
manual_prediction = manual_model.predict(X_test_ready)
print("Accuracy：", accuracy_score(y_test, manual_prediction))
print(classification_report(y_test, manual_prediction, zero_division=0)) # 未定義的指標以 0 顯示

形狀： (180, 12) (60, 12)
含缺失值： False False
Accuracy： 0.7166666666666667
              precision    recall  f1-score   support

          No       0.70      0.72      0.71        29
         Yes       0.73      0.71      0.72        31

    accuracy                           0.72        60
   macro avg       0.72      0.72      0.72        60
weighted avg       0.72      0.72      0.72        60



## 11｜改用 Pipeline 並比較基準模型

建立數值 Pipeline (中位數補值 → StandardScaler)、類別 Pipeline (眾數補值 → OneHotEncoder)。以 ColumnTransformer 分配欄位，再接 LogisticRegression(max_iter=1000)，存為 model。直接使用尚未補值的 X_train 訓練、X_test 預測。另訓練 DummyClassifier(strategy="most_frequent")，以表格比較兩者測試 Accuracy。

檢查：兩個模型都預測 60 筆；Pipeline 與第 10 題預測應一致。請輸出一致性檢查與前處理後特徵名稱。不能對 X_test 使用 fit_transform。

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier

### 各支線依順序處理資料
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)]) # 將欄位送入對應支線
model = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train) # fit 會依序學補值、縮放、編碼與模型參數
prediction = model.predict(X_test) # predict 自動套用既有規則再預測

### 與永遠猜訓練集多數類別的模型比較
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train) # 此策略忽略特徵內容，只統計答案
baseline_prediction = baseline.predict(X_test)
display(pd.DataFrame({"模型": ["多數類別基準", "完整 Pipeline"], "測試 Accuracy": [accuracy_score(y_test, baseline_prediction), accuracy_score(y_test, prediction)]}))
print("手動與 Pipeline 預測一致：", np.array_equal(manual_prediction, prediction))
print("特徵名稱：", model.named_steps["preprocessor"].get_feature_names_out())

,模型,測試 Accuracy
0,多數類別基準,0.516667
1,完整 Pipeline,0.716667


手動與 Pipeline 預測一致： True
特徵名稱： ['num__age' 'num__monthly_income' 'num__website_visits'
 'cat__city_Taichung' 'cat__city_Tainan' 'cat__city_Taipei'
 'cat__channel_Ads' 'cat__channel_Organic' 'cat__channel_Referral'
 'cat__membership_Basic' 'cat__membership_Gold' 'cat__membership_Silver']


## 12｜預測新客戶並儲存完整流程

建立下列 3 筆新資料，以第 11 題 model 預測。輸出 customer_id、預測結果、購買機率。購買機率需從 classes_ 找出 Yes 欄，不能假設固定位置。用 joblib 儲存完整 Pipeline 為 customer_pipeline.joblib，再讀回並確認預測一致。

| customer_id | age | monthly_income | website_visits | city | channel | membership |
|---|---:|---:|---:|---|---|---|
| NEW001 | 28 | 45000 | 8 | Taipei | Ads | Silver |
| NEW002 | 缺失 | 60000 | 2 | Kaohsiung | Referral | Gold |
| NEW003 | 45 | 缺失 | 5 | Tainan | 缺失 | Basic |

檢查：輸出 3 筆，機率介於 0 和 1；遇到缺失或未知城市不報錯；讀回預測一致。新客戶沒有實際購買答案，因此不要計算新資料 Accuracy。

In [ ]:
import joblib

### 建立新客戶資料
new_customers = pd.DataFrame({"customer_id": ["NEW001", "NEW002", "NEW003"], "age": [28, np.nan, 45], "monthly_income": [45000, 60000, np.nan], "website_visits": [8, 2, 5], "city": ["Taipei", "Kaohsiung", "Tainan"], "channel": ["Ads", "Referral", np.nan], "membership": ["Silver", "Gold", "Basic"]})
new_X = new_customers[feature_columns] # 使用訓練時相同的特徵欄位與單位
new_prediction = model.predict(new_X) # 自動補值、編碼、縮放後預測
classes = model.named_steps["classifier"].classes_
yes_column = np.flatnonzero(classes == "Yes")[0] # 找到 Yes 在機率矩陣的欄位位置
purchase_probability = model.predict_proba(new_X)[:, yes_column] # 每列客戶購買的估計機率
result = new_customers[["customer_id"]].copy()
result["預測結果"] = new_prediction
result["購買機率"] = purchase_probability
display(result.round(3))

### 儲存與讀回完整流程
joblib.dump(model, "customer_pipeline.joblib") # 包含前處理規則與分類模型
loaded_model = joblib.load("customer_pipeline.joblib") # 只讀取自己產生或信任來源的模型檔
print("讀回預測一致：", np.array_equal(new_prediction, loaded_model.predict(new_X)))

,customer_id,預測結果,購買機率
0,NEW001,Yes,0.678
1,NEW002,No,0.272
2,NEW003,No,0.386


讀回預測一致： True
